In [2]:
import pandas as pd
import numpy as np
import os
import transformers
import torch
import gc

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
from tqdm.notebook import tqdm
tqdm.pandas()



In [3]:


LLM_SAMPLE_SIZE = 1200
CATEGORIES = ["Мир", "Россия", "Экономика", "Наука и техника", "Спорт", "Культура"]

DATA_PATH = Path("news_data/cleaned_news_for_model.parquet")
df = pd.read_parquet(DATA_PATH)

df_model = df[["title", "text", "category_raw"]].dropna().copy()
df_model["llm_text"] = df_model["text"].astype(str) #.str[:800]  # обрезаем — LLM медленнее на длинных
df_model = df_model[df_model["llm_text"].str.len() > 0]

df_sample, _ = train_test_split(
    df_model,
    train_size=LLM_SAMPLE_SIZE,
    random_state=42,
    stratify=df_model["category_raw"]
)
# Один раз сохраняем выборку
df_sample.to_parquet("news_data/llm_sample_1200.parquet")

df_sample["category_raw"].value_counts()

category_raw
Мир                371
Россия             339
Экономика          263
Наука и техника     83
Спорт               81
Культура            63
Name: count, dtype: int64

In [5]:
CATEGORIES_STR = ", ".join(CATEGORIES)

SYSTEM_PROMPT = f"""Ты классификатор новостных статей.
Определи категорию текста и ответь ТОЛЬКО одним из вариантов: {CATEGORIES_STR}.
Никаких пояснений — только одно слово или фраза из списка."""

def classify_text(text, pipeline, max_new_tokens=20):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    output = pipeline(messages, max_new_tokens=max_new_tokens,do_sample=False)
    response = output[0]["generated_text"][-1]["content"].strip()

    # Ищем категорию в ответе модели
    for cat in CATEGORIES:
        if cat.lower() in response.lower():
            return cat
            
    print(f"[Неизвестно] Ответ модели: '{response}'")
    return "Неизвестно"  # если модель ответила не по формату

# Быстрый тест
test_text = "Сборная России по футболу провела тренировку перед матчем"
#print(classify_text(test_text, pipeline_qwen))

In [6]:
def free_memory(obj):
    del obj
    gc.collect()
    torch.mps.empty_cache()
    print("Память освобождена")

In [7]:
pipeline_gemma = transformers.pipeline(
     "text-generation",
     model="google/gemma-4-E4B-it",
     dtype="auto",
     device_map="auto",
)

print("Модель загружена")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Модель загружена


In [8]:
print(f"Классифицируем {len(df_sample)} текстов через Gemma...")
df_sample["pred_gemma"] = df_sample["llm_text"].progress_apply(
    lambda text: classify_text(text, pipeline_gemma)
)
df_sample["pred_gemma"].value_counts()

Классифицируем 1200 текстов через Gemma...


  0%|          | 0/1200 [00:00<?, ?it/s]

pred_gemma
Мир                529
Россия             225
Экономика          175
Наука и техника    114
Спорт               85
Культура            66
Неизвестно           6
Name: count, dtype: int64

In [9]:
free_memory (pipeline_gemma)

Память освобождена


In [12]:

pipeline_qwen = transformers.pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    dtype="auto",
    device_map="auto",
)

print("Модель загружена")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Модель загружена


In [13]:
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
from tqdm.notebook import tqdm
tqdm.pandas()

print(f"Классифицируем {len(df_sample)} текстов через Qwen...")
df_sample["pred_qwen"] = df_sample["llm_text"].progress_apply(
    lambda text: classify_text(text, pipeline_qwen)
)

df_sample["pred_qwen"].value_counts()

Классифицируем 1200 текстов через Qwen...


  0%|          | 0/1200 [00:00<?, ?it/s]

pred_qwen
Россия             408
Мир                358
Экономика          221
Спорт               98
Культура            68
Наука и техника     47
Name: count, dtype: int64

In [14]:

free_memory (pipeline_qwen)


Память освобождена


In [17]:
pipeline_llama = transformers.pipeline(
     "text-generation",
     model="meta-llama/Meta-Llama-3.1-8B-Instruct",
     dtype="auto",
     device_map="auto",
)

print("Модель загружена")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Модель загружена


In [18]:

print(f"Классифицируем {len(df_sample)} текстов через Llama...")
df_sample["pred_llama"] = df_sample["llm_text"].progress_apply(
    lambda text: classify_text(text, pipeline_llama)
)
df_sample["pred_llama"].value_counts()

Классифицируем 1200 текстов через Llama...


  0%|          | 0/1200 [00:00<?, ?it/s]

pred_llama
Мир                513
Россия             350
Экономика          114
Спорт               78
Наука и техника     76
Культура            65
Неизвестно           4
Name: count, dtype: int64

In [19]:
free_memory (pipeline_llama)

Память освобождена


In [20]:
def compute_metrics(y_true, y_pred, model_name):
    mask = y_pred != "Неизвестно"  # исключаем нераспознанные ответы
    y_true_f = y_true[mask]
    y_pred_f = y_pred[mask]

    accuracy    = accuracy_score(y_true_f, y_pred_f)
    macro_f1    = f1_score(y_true_f, y_pred_f, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true_f, y_pred_f, average="weighted", zero_division=0)
    unknown_rate = (~mask).mean()

    print(f"\n=== {model_name} ===")
    print(f"Accuracy:     {accuracy:.4f}")
    print(f"Macro F1:     {macro_f1:.4f}")
    print(f"Weighted F1:  {weighted_f1:.4f}")
    print(f"Неизвестно:   {unknown_rate:.1%} ответов не распознано")
    print()
    print(classification_report(y_true_f, y_pred_f, zero_division=0))

    return accuracy, macro_f1, weighted_f1, unknown_rate

acc_q, mf1_q, wf1_q, unk_q = compute_metrics(
    df_sample["category_raw"],
    df_sample["pred_qwen"],
    "Qwen2.5-7B-Instruct"
)

acc_l, mf1_l, wf1_l, unk_l = compute_metrics(
    df_sample["category_raw"],
    df_sample["pred_llama"],
    "Llama-3.1-8B-Instruct"
)
acc_g, mf1_g, wf1_g, unk_g = compute_metrics(
    df_sample["category_raw"],
    df_sample["pred_gemma"],
    "Gemma-4-E4B-it"
)


=== Qwen2.5-7B-Instruct ===
Accuracy:     0.7583
Macro F1:     0.7493
Weighted F1:  0.7565
Неизвестно:   0.0% ответов не распознано

                 precision    recall  f1-score   support

       Культура       0.69      0.75      0.72        63
            Мир       0.81      0.78      0.80       371
Наука и техника       0.89      0.51      0.65        83
         Россия       0.67      0.81      0.74       339
          Спорт       0.79      0.95      0.86        81
      Экономика       0.81      0.68      0.74       263

       accuracy                           0.76      1200
      macro avg       0.78      0.75      0.75      1200
   weighted avg       0.77      0.76      0.76      1200


=== Llama-3.1-8B-Instruct ===
Accuracy:     0.7140
Macro F1:     0.7358
Weighted F1:  0.7032
Неизвестно:   0.3% ответов не распознано

                 precision    recall  f1-score   support

       Культура       0.69      0.71      0.70        63
            Мир       0.67      0.92      

In [ ]:
results = pd.DataFrame([
    {
        "experiment": "08_qwen2.5_7b_instruct_zeroshot",
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "input": "text",
        "classifier": "zero-shot LLM",
        "sample_size": len(df_sample),
        "accuracy": acc_q,
        "macro_f1": mf1_q,
        "weighted_f1": wf1_q,
        "unknown_rate": unk_q,
    },
    {
        "experiment": "09_llama3.1_8b_instruct_zeroshot",
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "input": "text",
        "classifier": "zero-shot LLM",
        "sample_size": len(df_sample),
        "accuracy": acc_l,
        "macro_f1": mf1_l,
        "weighted_f1": wf1_l,
        "unknown_rate": unk_l,
    },
        {
        "experiment": "10_Gemma_4_E4B_it_instruct_zeroshot",
        "model": "google/gemma-4-E4B-it",
        "input": "text",
        "classifier": "zero-shot LLM",
        "sample_size": len(df_sample),
        "accuracy": acc_g,
        "macro_f1": mf1_g,
        "weighted_f1": wf1_g,
        "unknown_rate": unk_g,
    },
])

os.makedirs("../reports", exist_ok=True)
report_path = "../reports/llm_zeroshot_results.csv"

results.to_csv(
    report_path,
    mode="a",
    header=not os.path.exists(report_path),
    index=False
)

results